## Pytorch syntax


Familiarize einsums


In [23]:
import torch
import math
import torch.nn.functional as F 

A = torch.tensor([[1,2],[3,4]])
B = torch.tensor([[1,1],[1,1]])
C = torch.tensor([1,2,3,4])
D = torch.tensor([1,1,1,1])

# dot product - uv 
dot = torch.einsum('i,i->', C, D) # 2 inputs, both 1D denoted by i, input dimensions are i and outputs has no dimensions (scalar)

# outer product - uv^T
outer = torch.einsum('i,j -> ij', C, D) # CD^T

# matmul
matmul = torch.einsum('ac, cd -> ad', A, B)

# matmul with no summing
sumless_matmul = torch.einsum('ac,cd -> acd', A, B) # inner dimension that is hidden dim persists (2,2,2) where sumless[0][1][1] = A[0][1] * B[1][1] for example is the separated doc product of C[0][1]
sumless_matmul # multiplies the corresponding values in A and B, but does not sum them
summed_matmul = torch.einsum('acd->ad',sumless_matmul)
summed_matmul

# batched matmul
input = torch.rand(32, 1024, 64) # normal batch size embedding dim of 64
hidden = torch.rand(64, 256)
output = torch.einsum('ijk, kl -> ijl', input, hidden)
output.shape # torch.Size([32, 1024, 256]), nice!

# full quadratic attention
query = torch.rand(32, 1024, 256)
key = torch.rand(32, 1024, 256)
value = torch.rand(32, 1024, 256)

qk = torch.einsum('bqd, bkd -> bqk', query, key)
d = query.shape[-1]
softmax_qk = F.softmax(qk/math.sqrt(d), dim=-1) # softmax, consuming the k dimension (so row-wise softmax)
attention = torch.einsum('bqk, bkd -> bkd', softmax_qk, value)

# without einsums
qk = query @ key.transpose(-2, -1) / math.sqrt(query.shape[-1])
B, T, T = qk.shape
mask = torch.tril(torch.ones(T, T)).bool()
print(mask.shape, qk.shape)
masked_qk = qk.masked_fill(~mask, float('-inf')) # where the mask is False, put -inf
print(masked_qk)

qkv = F.softmax(qk, dim=-1) @ value
qkv = qk @ value # (B, T, T) @ (T, d) = (B, T, d)

assert attention.shape == qkv.shape

torch.Size([1024, 1024]) torch.Size([32, 1024, 1024])
tensor([[[3.9365,   -inf,   -inf,  ...,   -inf,   -inf,   -inf],
         [4.0743, 3.8907,   -inf,  ...,   -inf,   -inf,   -inf],
         [4.0541, 3.7908, 3.9739,  ...,   -inf,   -inf,   -inf],
         ...,
         [4.0011, 3.9469, 3.9740,  ..., 4.0106,   -inf,   -inf],
         [4.0143, 3.6939, 4.1044,  ..., 4.0001, 4.0864,   -inf],
         [4.1160, 3.8555, 4.3491,  ..., 4.2996, 4.2417, 4.0130]],

        [[4.2850,   -inf,   -inf,  ...,   -inf,   -inf,   -inf],
         [4.1208, 4.1571,   -inf,  ...,   -inf,   -inf,   -inf],
         [4.1241, 4.0620, 3.6656,  ...,   -inf,   -inf,   -inf],
         ...,
         [4.0660, 4.0712, 3.9484,  ..., 4.1646,   -inf,   -inf],
         [3.9658, 4.0996, 3.7159,  ..., 3.9716, 3.7293,   -inf],
         [4.1847, 4.1687, 4.0127,  ..., 4.1992, 4.0653, 4.4096]],

        [[4.2618,   -inf,   -inf,  ...,   -inf,   -inf,   -inf],
         [4.2777, 3.7168,   -inf,  ...,   -inf,   -inf,   -inf],
    

Linear attention


Gather operation

Practice gathering logits - let's code the CISPO algorithm, using stable softmax with k3 divergence.


In [ ]:
import hf

hf.fromPretrained()
hf.CausalLM()